# MRP / 04 — Conditional reversal after large bars: discovery 2020-01-01 – 2023-06-30

Protocol: [`MRP 003`](../../discussions/MNQ_Return_Prediction/003_claude_mrp02_conditional_reversal_preregistration.md). All results in **ticks per contract** (1 tick = 0.25 points = $0.50).

- Event: |z_t| ≥ k (z from the MRP 001 volatility model). Trade against the bar, entry at the close of t + d, exit h bars later, no overlap.
- Grid: k ∈ {1.5, 2, 2.5, 3, 4} × d ∈ {0, 1} × h ∈ {1, 2, 3, 6} × session ∈ {all, Asia, London, NY, post} = 200.
- Costs: **market** = 1 tick + 2 × commission (verdict; default $0.50/side → 3 ticks) · **realized** = 1.6 ticks (Iván's experience, reported).
- Gates: D1 n ≥ 150 · D2 net > 0 at market cost, day-clustered t, BH q ≤ 0.05 over 200 · D3 net > 0 in ≥ 3 of 4 periods · D4 gross > 0 for both longs (after down bars) and shorts (after up bars).
- Candidates are frozen to `04/candidates.csv` (SHA-256 in `04/summary.json`).

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("MRP_SYNTHETIC") == "1"
if SYNTHETIC:   # separate folder: the reversal smoke data has a planted reversal and no FX effect
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("MRP_REV_SYNTH_DIR") or tempfile.mkdtemp(prefix="mrp_rev_")
    os.environ["MRP_REV_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
D = dt.date
OUT = Path("04"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", P.MRP_VERSION, V.REV_VERSION, "| synthetic" if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    P.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.test[1], CFG, beta_jpy=0.0,
                      reversal_points=6.0, tails_df=4)


In [ ]:
RC = V.RevConfig()
if SYNTHETIC:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2020, 7, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020H1": (D(2020, 1, 1), D(2020, 6, 30)), "2020H2": (D(2020, 7, 1), D(2020, 12, 31))}
else:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2021, 1, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020": (D(2020, 1, 1), D(2020, 12, 31)), "2021": (D(2021, 1, 1), D(2021, 12, 31)),
               "2022": (D(2022, 1, 1), D(2022, 12, 31)), "2023H1": (D(2023, 1, 1), D(2023, 6, 30))}
grid = P.load_grid(CFG.warmup_start, END, CFG)
sig2 = V.event_sigma(grid, FIRST_FIT, REFITS, END, CFG)
bt = V.bar_table(grid, sig2, CFG)
bt = bt[(bt["trade_date"] >= CFG.train[0]) & (bt["trade_date"] <= END)]
print(len(bt), "bars; market cost", RC.market_cost_ticks, "ticks; realized", RC.realized_cost_ticks, "ticks")
e = bt[bt["eligible"]]
cnt = pd.DataFrame({f"|z|>={k:g}": (e["z"].abs() >= k).groupby(pd.to_datetime(e["trade_date"]).dt.year).mean() * 100
                    for k in RC.ks}).round(2)
print("share of eligible bars that are events (%), by year:"); display(cnt)
print("median 1-bar move in ticks:", float((np.abs(e["ret"]) * e["close"] / V.TICK).median()))

## 1. Size curve (descriptive): mean reversal in ticks by |z| bin, every eligible bar, h = 1

In [ ]:
curves = {d: V.size_curve(bt, RC, d=d, h=1) for d in RC.delays}
for d, c in curves.items():
    print(f"d = {d}:")
    display(c.pivot(index="bin", columns="session", values="mean_rev_ticks").round(2))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for a, (d, c) in zip(ax, curves.items()):
    for s in RC.sessions:
        cc = c[c["session"] == s]
        a.plot(range(len(cc)), cc["mean_rev_ticks"], "o-", label=s)
    a.set_xticks(range(len(cc))); a.set_xticklabels(cc["bin"], rotation=30)
    a.axhline(RC.market_cost_ticks, color="r", ls="--", lw=.8, label="market cost")
    a.axhline(RC.realized_cost_ticks, color="orange", ls=":", lw=.8, label="realized cost")
    a.axhline(0, color="k", lw=.6); a.set_title(f"entry delay d = {d}, hold 1 bar"); a.set_xlabel("|z| of the large bar")
ax[0].set_ylabel("mean move against the bar (ticks)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "size_curve.png", dpi=110); plt.show()

## 2. Scan of the 200 configurations

In [ ]:
df, store = V.scan(bt, RC, PERIODS)
cols = ["config", "n", "gross", "net", "t", "q_bh", "net_realized", "gross_long", "gross_short", "win_rate", "mean_win",
        "mean_loss", "q05", "worst", "max_dd", "periods_positive", "D1", "D2", "D3", "D4", "candidate", "realized_only"]
print("top 25 by net ticks at market cost (D1 only):")
display(df[df["D1"]].sort_values("net", ascending=False)[cols].head(25).round(3))
print(f"candidates: {int(df['candidate'].sum())} | clear realized cost only: {int(df['realized_only'].sum())}")
df.to_csv(OUT / "scan.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].hist(df.loc[df["D1"], "net"], bins=40); ax[0].axvline(0, color="k"); ax[0].set_title("net ticks at market cost (D1 configs)")
ax[0].set_xlabel("ticks per trade")
top = df[df["D1"]].sort_values("net", ascending=False).head(5)["config"]
for c in top:
    tr = store[c]
    ax[1].plot(pd.to_datetime(tr["trade_date"]), (tr["gross_ticks"] - RC.market_cost_ticks).cumsum(), label=c)
ax[1].axhline(0, color="k", lw=.6); ax[1].legend(fontsize=7); ax[1].set_title("cumulative net ticks, top 5 by net")
plt.tight_layout(); plt.savefig(OUT / "scan.png", dpi=110); plt.show()

## 3. Freeze candidates

In [ ]:
cands = df[df["candidate"]][["config", "k", "d", "h", "session", "n", "net", "t", "q_bh"]]
display(cands)
sha = V.freeze(cands, OUT / "candidates.csv")
summary = {"engines": [P.MRP_VERSION, V.REV_VERSION], "synthetic": SYNTHETIC, "market_cost_ticks": RC.market_cost_ticks,
           "realized_cost_ticks": RC.realized_cost_ticks, "n_candidates": int(len(cands)), "candidates_sha256": sha,
           "realized_only": df.loc[df["realized_only"], "config"].tolist(),
           "best_net_market": df[df["D1"]].sort_values("net", ascending=False).head(10)[["config", "n", "net", "t", "q_bh"]].to_dict("records"),
           "size_curve": {str(d): c.to_dict("records") for d, c in curves.items()}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=float))
print("frozen", sha)